In [0]:
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import mlflow
import mlflow.spark
import mlflow.data
from mlflow.models.signature import infer_signature

import os

# ===============================
# CONFIG MLflow (UC compatible)
# ===============================
TMP_PATH = "/Volumes/iotmlhealthcatalog/default/iotbatch/tmp"
os.environ["MLFLOW_DFS_TMP"] = TMP_PATH

# ===============================
# 1. LOAD
# ===============================
gold_df = spark.table("iotmlhealthcatalog.gold.vitaldbtrain")

features_list = [
    "sbp",
    "hr",
    "spo2",
    "temp",
    "shock_index",
    "hr_spo2_ratio",
    "is_low_sbp",
    "is_high_hr",
    "is_low_spo2"
]

gold_df = gold_df.select(
    "caseid", "timestamp", "target",
    *features_list
)

# ===============================
# 2. CLEAN (IMPORTANT)
# ===============================
gold_df = gold_df.dropna(subset=["target"])

# ===============================
# 2. SPLIT TEMPOREL
# ===============================
gold_df = gold_df.orderBy("timestamp")

split_index = int(gold_df.count() * 0.8)

train_data = gold_df.limit(split_index)
test_data = gold_df.subtract(train_data)

print("Train count:", train_data.count())
print("Test count :", test_data.count())


# ===============================
# 4. FEATURE PIPELINE
# ===============================
# Assembler
assembler = VectorAssembler(
    inputCols=features_list,
    outputCol="features",
    handleInvalid="keep"   # meilleur que skip
)

# Modèle
rf = RandomForestClassifier(
    labelCol="target",
    featuresCol="features",
    numTrees=80,
    maxDepth=6,
    minInstancesPerNode=30,
    seed=42
)

# ===============================
# 5. MLflow
# ===============================
mlflow.set_experiment("/Users/<your-user-email>/iot_predict_hypertension")

# Verifier si un un MLflow est déjà actif
if mlflow.active_run():
    mlflow.end_run()

# ===============================
# 6. TRAINING BLOCK
# ===============================
with mlflow.start_run(run_name="RF_VitalDB_For_Streaming") as run:

    pipeline = Pipeline(stages=[assembler, rf])

    model = pipeline.fit(train_data)

    predictions = model.transform(test_data)

    # ===============================
    # 7. ÉVALUATION
    # ===============================
    evaluator = BinaryClassificationEvaluator(
        labelCol="target",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    auc = evaluator.evaluate(predictions)

    evaluator_pr = BinaryClassificationEvaluator(
        labelCol="target",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )

    pr_auc = evaluator_pr.evaluate(predictions)

    # ===============================
    # 8. LOGGING MLflow
    # ===============================
    mlflow.log_params({
        "numTrees": 80,
        "maxDepth": 6,
        "minInstancesPerNode": 30,
        "features_count": len(features_list)
    })

    mlflow.log_metrics({
        "AUC_ROC": auc,
        "AUC_PR": pr_auc
    })

    # ===============================
    # 9. SIGNATURE
    # ===============================
    sample_input = train_data.select(features_list).limit(100).toPandas()
    sample_output = model.transform(train_data.limit(100)) \
        .select("prediction") \
        .toPandas()

    signature = infer_signature(sample_input, sample_output)

    # ===============================
    # 10. SAVE MODEL
    # ===============================
    mlflow.spark.log_model(
        model,
        "rf_pipeline_model",
        signature=signature,
        dfs_tmpdir=TMP_PATH
    )

    mlflow.log_input(
        mlflow.data.from_spark(train_data.select(features_list).limit(100)),
        context="training"
    )

    # ===============================
    # 9. OUTPUT FINAL
    # ===============================
    run_id = run.info.run_id
    print("AUC ROC:", auc)
    print("AUC PR :", pr_auc)
    print("Run ID:", run_id)